In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


def get_proj_root():
    
    curr_path = Path(__file__).resolve()
    
    for parent in [curr_path] + list(curr_path.parents):
        
        if (parent / ".git").exists() or (parent / "requirements.txt").exists():
            return parent
        
    raise FileNotFoundError("could not locate project root directory")


def parse_rewards_file(filepath):
    """Parses the text file to extract episode rewards using regex."""

    with open(filepath, 'r') as file:
        content = file.read()
    
    # Regex to find all reward values. Matches: 'Episode X': <value>
    # captures positive, negative, integer, and float numbers
    rewards = re.findall(r"'Episode \d+':\s*([-+]?[0-9]*\.?[0-9]+)", content)

    return [float(r) for r in rewards]


def load_data(root_dir):
    """Traverses the directory structure and compiles data into a DataFrame."""

    data = []
    root_path = Path(root_dir)
    
    # Check if directory exists
    if not root_path.exists():
        raise FileNotFoundError(f"Directory {root_dir} not found.")

    # Walk through all directories and files
    for filepath in root_path.rglob('*'):

        if filepath.is_file():
                
            # Determine Model Type from parent directories
            if "Active Models" in filepath.parts:
                model_type = "Active"

            elif "Baseline Models" in filepath.parts:
                model_type = "Baseline"

            else:
                continue # Skip files not in these directories
                
            # Determine Level from parent directories (e.g., "Level 1", "Level 2")
            level_match = re.search(r'Level (\d+)', str(filepath))
            level = f"Level {level_match.group(1)}" if level_match else "Unknown Level"
            
            # Determine Percentage Reduction from filename (only for Active models)
            # Looks for one of the specific percentages in the filename

            reduction = None

            if model_type == "Active":
                reduction_match = re.search(r'(100|80|60|40|20|10|5|1)(?=\D|$)', filepath.stem)

                if reduction_match:
                    reduction = int(reduction_match.group(1))
            
            # Extract rewards
            rewards = parse_rewards_file(filepath)
            
            # Append each episode's data
            for ep_num, reward in enumerate(rewards):

                data.append({
                    'Model_Type': model_type,
                    'Level': level,
                    'Reduction': reduction,
                    'Episode': ep_num,
                    'Reward': reward,
                    'Filename': filepath.name
                })
                
    return pd.DataFrame(data)


def generate_plots(df):
    """Generates and saves the required visualizations."""
    
    # Set the visual style
    sns.set_theme(style="whitegrid")
    
    # ---------------------------------------------------------
    # Graph 1: Overall Differences (Active vs Baseline)
    # ---------------------------------------------------------
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df, x='Model_Type', y='Reward', palette='Set2')

    plt.title('Overall Performance: Active vs Baseline Models', fontsize=14)
    plt.ylabel('Episode Reward', fontsize=12)

    plt.xlabel('Model Type', fontsize=12)
    plt.tight_layout()

    plt.savefig('1_overall_active_vs_baseline.png')
    plt.show()
    
    # ---------------------------------------------------------
    # Graph 2: Active vs Baseline Across Different Levels
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))

    # Sort levels alphabetically so Level 1 comes before Level 2
    df_sorted_levels = df.sort_values('Level') 

    sns.boxplot(data=df_sorted_levels, x='Level', y='Reward', hue='Model_Type', palette='Set2')

    plt.title('Performance Across Levels: Active vs Baseline', fontsize=14)
    plt.ylabel('Episode Reward', fontsize=12)

    plt.xlabel('Level', fontsize=12)
    plt.legend(title='Model Type')

    plt.tight_layout()
    plt.savefig('2_performance_across_levels.png')

    plt.show()

    # ---------------------------------------------------------
    # Graph 3: Effect of Reductions on Active Models
    # ---------------------------------------------------------
    # Filter only Active models that have a valid reduction mapped
    active_df = df[(df['Model_Type'] == 'Active') & (df['Reduction'].notna())].copy()
    
    if not active_df.empty:
        plt.figure(figsize=(12, 6))
        
        # Sort by reduction descending (100 -> 1)
        active_df = active_df.sort_values(by=['Level', 'Reduction'], ascending=[True, False])
        
        # use a pointplot to show the trend of the mean rewards as reductions decrease
        sns.pointplot(data=active_df, x='Reduction', y='Reward', hue='Level', 
                      dodge=True, markers=['o', 's', 'D'], capsize=.1, err_kws={'linewidth': 1})
        
        # Invert x-axis so it goes 100 -> 80 -> 60 ... -> 1
        plt.gca().invert_xaxis()
        
        plt.title('Effect of Percentage Reduction on Active Models by Level', fontsize=14)
        plt.ylabel('Mean Episode Reward (with 95% CI)', fontsize=12)

        plt.xlabel('Percentage Reduction (%)', fontsize=12)
        plt.legend(title='Level')

        plt.tight_layout()
        plt.savefig('3_reduction_effects_trend.png')

        plt.show()

    else:
        print("No valid reduction data found in Active model filenames to plot Graph 3.")


if __name__ == "__main__":
    
    
    ROOT_DIR = get_proj_root()
    print(f"\n\nProject root directory: {ROOT_DIR}")

    MODEL_DATA_PATH = f"{ROOT_DIR}/model_performance_data"
    
    print("Loading and parsing data...")
    df = load_data(MODEL_DATA_PATH)
    
    if df.empty:
        print("No data was loaded. Please check your directory structure and file contents.")

    else:
        print(f"Successfully loaded {len(df)} episodes across {df['Filename'].nunique()} files.")
        print("Generating graphs...")
        
        generate_plots(df)
        print("Graphs generated and saved to the current directory.")